In [ ]:
# fr/data-analysis/normal/04-filtering-rows
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("titanic.csv", ())


## Filtrer avec des conditions booléennes

Le filtrage vous permet de vous concentrer sur le sous-ensemble de données qui compte. Vous créez un **masque booléen** — une Series de valeurs True/False — et vous l'utilisez pour sélectionner des lignes.


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")

# Filter passengers older than 30
older = df[df["Age"] > 30]
print(older.shape)   # fewer rows than original 891


L'expression `df["Age"] > 30` produit une Series booléenne :


In [ ]:
0       True
1       True
2      False
3       True
...


La passer à l'intérieur de `df[...]` ne conserve que les lignes où la valeur est `True`.

## Combiner des conditions

Utilisez `&` (et) et `|` (ou) pour combiner des conditions. **Chaque condition doit être entourée de parenthèses :**


In [ ]:
# Female passengers in first class
first_class_female = df[(df["Sex"] == "female") & (df["Pclass"] == 1)]
print(first_class_female.head())


In [ ]:
# Passengers younger than 25 OR older than 60
young_or_old = df[(df["Age"] < 25) | (df["Age"] > 60)]
print(young_or_old.shape)


Erreur courante : utiliser `and`/`or` au lieu de `&`/`|`. Les opérateurs `and`/`or` de Python ne fonctionnent pas élément par élément sur les Series pandas et provoqueront une erreur.

## Utiliser .isin() pour plusieurs valeurs

Lorsque vous devez comparer une liste de valeurs, utilisez `.isin()` :


In [ ]:
# Passengers who embarked from Cherbourg or Southampton
embarked_filter = df[df["Embarked"].isin(["C", "S"])]


In [ ]:
# Passengers in class 1 or 2
upper_classes = df[df["Pclass"].isin([1, 2])]


## Utiliser .between() pour les plages

La méthode `.between()` est plus propre que l'enchaînement de deux comparaisons :


In [ ]:
# Passengers aged 20 to 30 (inclusive by default)
twenties = df[df["Age"].between(20, 30)]
print(twenties.shape)


Cela équivaut à `df[(df["Age"] >= 20) & (df["Age"] <= 30)]` mais c'est plus lisible.

## Filtrer avec des méthodes de chaîne

L'accesseur `.str` permet d'appliquer des opérations sur les chaînes à toute une colonne :


In [ ]:
# Passengers whose name contains "Master" (a title)
masters = df[df["Name"].str.contains("Master", na=False)]
print(masters.shape)


In [ ]:
# Passengers whose ticket starts with "A"
a_tickets = df[df["Ticket"].str.startswith("A", na=False)]


Le paramètre `na=False` gère les valeurs manquantes avec élégance — sans lui, les entrées NaN provoqueraient des erreurs.

## Filtrer avec .query()

Pour les filtres complexes, `.query()` offre une alternative lisible :


In [ ]:
# Equivalent to df[(df["Age"] > 25) & (df["Survived"] == 1)]
survivors_over_25 = df.query("Age > 25 and Survived == 1")


Cela se lit presque comme de l'anglais et évite la syntaxe répétitive `df["colonne"]`.

## Stocker les filtres dans des variables

Pour les conditions complexes, stockez d'abord le masque booléen dans une variable :


In [ ]:
is_female = df["Sex"] == "female"
is_first_class = df["Pclass"] == 1
is_survived = df["Survived"] == 1

# Combine them
result = df[is_female & is_first_class & is_survived]
print(f"Female first-class survivors: {len(result)}")


Cette approche rend votre code beaucoup plus facile à lire et à déboguer.

## Essayez-le

À partir du jeu de données Titanic, filtrez pour trouver :
1. Tous les passagers qui ont payé plus de 100 de tarif
2. Toutes les passagères femmes en troisième classe
3. Tous les passagers dont le nom contient le titre « Mrs »


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")

high_fare = df[df["Fare"] > 100]
print(f"High fare passengers: {len(high_fare)}")

third_class_female = df[(df["Sex"] == "female") & (df["Pclass"] == 3)]
print(f"Third-class females: {len(third_class_female)}")

mrs = df[df["Name"].str.contains("Mrs", na=False)]
print(f"Passengers with title Mrs: {len(mrs)}")


## Points clés à retenir

- L'indexation booléenne `df[mask]` est le principal mécanisme de filtrage dans pandas
- Utilisez `&` pour ET, `|` pour OU — entourez toujours les conditions individuelles de parenthèses
- `.isin()` compare une liste ; `.between()` gère proprement les plages
- `.str.contains()` filtre par correspondance de sous-chaîne — utilisez `na=False` par sécurité

## Défi pratique

À partir du jeu de données Titanic, trouvez tous les passagers qui : (1) étaient de sexe masculin, (2) voyageaient en deuxième ou troisième classe, (3) avaient entre 18 et 35 ans, et (4) ont survécu. Combien de passagers satisfont les quatre conditions ?


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
